# Vanilla diffusion versus statistical forecasts — execution pilot

This notebook tests the comparison infrastructure on the unchanged **synthetic Gaussian** configuration.
It runs two of the configured 50 epochs, then samples only the first **64 validation dates**, with **256 scenarios**.
It does not establish a diffusion advantage or complete the scientific experiment.

The best checkpoint is selected using validation loss. Scores and bootstrap intervals on the same validation
period are **development evidence, not independent final generalization or selection-adjusted significance**.
The held-out test partition is never evaluated. Mean-reverting full-training commands are provided as Markdown only.

Existing full baseline archives can be reused. Otherwise the notebook generates a small matched statistical baseline.
No CDE, geometry, topology, Levy, portfolio or multi-seed neural experiments are included.

## Kaggle setup

Enable Internet and a GPU for the configured pilot. CPU also works, but diffusion inference can be slower.
The established synchronization cell selects `feature/synthetic-market`. Dependencies use the existing project extras.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Regression tests

These exercise tiny CPU checkpoints, strict archive alignment, per-date scores and paired block bootstrap.
The tests do not run full scientific training. New synthetic training records `data_identity.json` beside existing checkpoints.

In [ ]:
!pytest -q tests/unit/test_synthetic_diffusion_comparison.py tests/unit/test_refactor_compatibility.py tests/unit/test_probabilistic_evaluation.py
assert _exit_code == 0, "Forecast comparison regression tests failed."

## Execution pilot — two epochs

This dedicated directory is separate from full research runs. `--stop-after-epoch 2` is an absolute boundary;
`--resume` saves/restores `last.pt`. Re-execution does not train another two epochs.
The source config, optimizer, architecture, schedule, loss, seeds and training-only scaler are unchanged.

In [ ]:
from pathlib import Path
from diffusion_portfolio.config import load_config
from diffusion_portfolio.utils.research_checks import check_checkpoint_pair

RUN_DIR = Path("/kaggle/working/synthetic_diffusion/gaussian_execution_pilot")
if (RUN_DIR / "config.yaml").exists():
    assert load_config(RUN_DIR / "config.yaml") == load_config("configs/synthetic_gaussian.yaml"), "Existing run has a different configuration."
    assert (RUN_DIR / "data_identity.json").is_file(), "Legacy run lacks training-time source evidence; use a new pilot directory."
check_checkpoint_pair(RUN_DIR)

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/synthetic_gaussian.yaml \
    --output-dir /kaggle/working/synthetic_diffusion/gaussian_execution_pilot \
    --device auto --resume --stop-after-epoch 2
assert _exit_code == 0, "Two-epoch execution pilot failed."

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt

for name in ("best.pt", "last.pt", "standardizer.npz", "config.yaml", "data_identity.json"):
    assert (RUN_DIR / name).is_file(), f"Missing pilot prerequisite: {name}"
check_checkpoint_pair(RUN_DIR)
checkpoint = torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)
history = pd.DataFrame(checkpoint["history"])
display(history)
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="Gaussian Vanilla — execution pilot")
ax.set(xlabel="Epoch", ylabel="Diffusion loss")
ax.legend(["Train Loss", "Validation Loss"])
ax.grid(alpha=.3)
plt.show()
print("Completed epochs:", checkpoint["epoch"], "/", load_config(RUN_DIR / "config.yaml").training.epochs)

## Limited validation inference

Only 64 windows enter the inference DataLoader. The saved scaler and best-validation checkpoint are restored.
The destination must be empty or absent: choose a new destination for another inference run.
Metadata records the checkpoint, source fingerprints, epoch budget, subset, seed, batch size and device.
Individual stochastic samples can change with batch size, last-batch size, numerical libraries or CPU/GPU.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/synthetic_diffusion/gaussian_execution_pilot \
    --split validation --limit 64 --batch-size 16 --device auto \
    --output-dir /kaggle/working/synthetic_comparisons/gaussian_pilot_inference
assert _exit_code == 0, "Limited validation inference failed."

## Reuse or generate matched statistical forecasts

Set `BASELINE_DIR` to an existing Stage 4E-1 artifact directory containing `report.json`, resolved config,
scaler and per-model archives. Full 1,500-date archives are supported: comparison uses their matching prefix.
No particular Kaggle input path is assumed. The tool will reject incompatible source fingerprints or scenario counts.

Leave it as `None` to generate only 64 targets using the pilot's saved config and 256 scenarios. Existing files are never replaced.

In [ ]:
BASELINE_DIR = None  # Optional: Path("/your/existing/gaussian_full_baseline")

if BASELINE_DIR is None:
    BASELINE_DIR = Path("/kaggle/working/synthetic_comparisons/gaussian_matched_baseline")
    !PYTHONUNBUFFERED=1 python -u scripts/benchmark_synthetic_forecasts.py --config /kaggle/working/synthetic_diffusion/gaussian_execution_pilot/config.yaml --models gaussian,ar1,ewma --split validation --n-scenarios 256 --limit 64 --seed 42 --output-dir {BASELINE_DIR}
    assert _exit_code == 0, "Matched statistical benchmark failed."
else:
    BASELINE_DIR = Path(BASELINE_DIR)
    assert (BASELINE_DIR / "report.json").is_file(), "Existing baseline report is missing."

## Paired scores and exploratory uncertainty

Differences are **diffusion minus baseline**: negative favors diffusion. Moving-block bootstrap resamples
paired chronological score differences, using overlapping blocks of 20 forecast observations and 1,000 replications.
It assumes approximately stationary differences and an adequate block length. With only 64 dates, uncertainty is
poorly characterized. These intervals do not account for checkpoint selection, training uncertainty or repeated
Monte Carlo forecast sampling. Calibration is reported separately from proper scores.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/compare_synthetic_forecasts.py \
    --baseline-dir {BASELINE_DIR} \
    --diffusion-dir /kaggle/working/synthetic_comparisons/gaussian_pilot_inference \
    --limit 64 --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
    --output-dir /kaggle/working/synthetic_comparisons/gaussian_pilot_comparison
assert _exit_code == 0, "Strict paired forecast comparison failed."

In [ ]:
import json

COMPARISON_DIR = Path("/kaggle/working/synthetic_comparisons/gaussian_pilot_comparison")
report = json.loads((COMPARISON_DIR / "comparison_report.json").read_text())
print("Training status:", report["training_run"])
print(report["interpretation"])
paired = pd.read_csv(COMPARISON_DIR / "paired_comparison.csv")
display(paired)
display(pd.DataFrame([
    {"model": name, "crps": values["crps_mean"], "energy_score": values["energy_score"], "mean_abs_ace": values["mean_abs_ace"]}
    for name, values in report["models"].items()
]))
display(pd.read_csv(COMPARISON_DIR / "calibration.csv"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, metric in zip(axes, ("crps", "energy_score")):
    rows = paired.loc[paired.metric == metric].reset_index(drop=True)
    ax.hlines(rows.index, rows.ci95_lower, rows.ci95_upper, color="steelblue", linewidth=2)
    ax.scatter(rows.mean_difference, rows.index, color="black", zorder=3)
    ax.axvline(0, color="gray", linestyle="--")
    ax.set(yticks=rows.index, yticklabels=rows.baseline, xlabel="Diffusion minus baseline", title=f"{metric}: pilot mean and block interval")
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

## Meaningful training — manual commands only

After reviewing the pilot, run the ordinary 50-epoch budgets in separate experiment directories.
These commands are intentionally Markdown, so Run All does not execute them.

```bash
python -u scripts/train_diffusion.py --config configs/synthetic_gaussian.yaml --output-dir /kaggle/working/synthetic_diffusion/gaussian_full --device cuda --resume
python -u scripts/train_diffusion.py --config configs/synthetic_mean_reverting.yaml --output-dir /kaggle/working/synthetic_diffusion/mean_reverting_full --device cuda --resume
```

For a completed run, use `evaluate_diffusion.py --split validation --output-dir <new-directory>` without `--limit`
for all 1,500 dates, then compare against matching 256-scenario baseline archives. Full inference is not launched here.
A completed declared budget does not imply convergence or forecast superiority. Validation-selected comparisons
remain development evidence; locked independent testing requires a later protocol decision.

## Save artifacts

Preserve the pilot run (including `data_identity.json`), inference directory and comparison reports with the Kaggle version.
Comparison writes `comparison_report.json`, `per_date_scores.csv`, `paired_comparison.csv` and `calibration.csv`.
Save any newly generated matched baseline separately. There are no automatic central-dataset uploads.
Missing training-time source evidence is a hard comparison failure; old checkpoints are never silently certified or repaired.
Historical KF12/FF49/FF100 and previous baseline artifacts are untouched.